In [ ]:
# -*- coding: utf-8 -*-
import dpdata, random
from pathlib import Path

# ====== config ======
ROOT = Path(r"out_v1/deepmd_npy")  # thư mục hiện có Si56C56H52, Si56C56H55, Si56C56H58
SPLIT = (0.8, 0.1, 0.1)            # train, val, test
SEED  = 42                         # để tái lập kết quả
# ====================

random.seed(SEED)

def detect_fmt(system_dir: Path) -> str:
    """Tự động nhận diện deepmd/raw hay deepmd/npy."""
    # raw: các file text .raw (type.raw/coord.raw/energy.raw/force.raw/virial.raw/type_map.raw)
    raw_files = ["type.raw", "coord.raw", "energy.raw", "force.raw", "virial.raw", "type_map.raw"]
    if all((system_dir / f).exists() for f in raw_files):
        return "deepmd/raw"
    # npy: các file .npy
    npy_files = ["type.npy", "coord.npy", "energy.npy", "force.npy", "virial.npy", "type_map.npy"]
    if all((system_dir / f).exists() for f in npy_files):
        return "deepmd/npy"
    # đôi khi thiếu virial -> kiểm tra tối thiểu
    if (system_dir / "type.raw").exists() and (system_dir / "coord.raw").exists():
        return "deepmd/raw"
    if (system_dir / "type.npy").exists() and (system_dir / "coord.npy").exists():
        return "deepmd/npy"
    raise RuntimeError(f"Không nhận diện được định dạng DeepMD trong: {system_dir}")

def split_indices(n, ratios):
    """Trả về (idx_train, idx_val, idx_test) theo tỉ lệ ratios trên 0..n-1."""
    idx = list(range(n))
    random.shuffle(idx)
    r_train, r_val, r_test = ratios
    n_train = int(n * r_train)
    n_val   = int(n * r_val)
    # đảm bảo không bỏ sót frame
    idx_train = idx[:n_train]
    idx_val   = idx[n_train:n_train+n_val]
    idx_test  = idx[n_train+n_val:]
    # edge cases: nếu quá ít frame, đẩy hết vào train
    if not idx_train and n > 0:
        idx_train, idx_val, idx_test = idx, [], []
    return idx_train, idx_val, idx_test

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def main():
    # liệt kê các composition folders (bỏ qua nếu là train/val/test đã tồn tại)
    compositions = [p for p in ROOT.iterdir()
                    if p.is_dir() and p.name.lower() not in {"train", "val", "test"}]

    if not compositions:
        print(f"[WARN] Không tìm thấy thư mục composition trong {ROOT}")
        return

    # tạo đích
    out_train = ensure_dir(ROOT / "train")
    out_val   = ensure_dir(ROOT / "val")
    out_test  = ensure_dir(ROOT / "test")

    total = {"train": 0, "val": 0, "test": 0}

    for comp in sorted(compositions, key=lambda x: x.name):
        try:
            fmt = detect_fmt(comp)
            ls = dpdata.LabeledSystem(str(comp), fmt=fmt)
        except Exception as e:
            print(f"[WARN] Bỏ qua {comp.name}: {e}")
            continue

        n = ls.get_nframes()
        idx_train, idx_val, idx_test = split_indices(n, SPLIT)

        # cắt theo frame
        ls_tr = ls.sub_system(idx_train) if idx_train else None
        ls_va = ls.sub_system(idx_val)   if idx_val   else None
        ls_te = ls.sub_system(idx_test)  if idx_test  else None

        # ghi ra đúng định dạng ban đầu (raw hoặc npy)
        if ls_tr is not None:
            target = out_train / comp.name
            ensure_dir(target)
            ls_tr.to(fmt, str(target))
            total["train"] += len(idx_train)

        if ls_va is not None:
            target = out_val / comp.name
            ensure_dir(target)
            ls_va.to(fmt, str(target))
            total["val"] += len(idx_val)

        if ls_te is not None:
            target = out_test / comp.name
            ensure_dir(target)
            ls_te.to(fmt, str(target))
            total["test"] += len(idx_test)

        print(f"[OK] {comp.name}: frames={n} -> "
              f"train={len(idx_train)}, val={len(idx_val)}, test={len(idx_test)} (fmt={fmt})")

    print("[SUMMARY] total frames written:",
          f"train={total['train']}, val={total['val']}, test={total['test']}")
    print(f"[DONE] Output at: {ROOT / 'train'}, {ROOT / 'val'}, {ROOT / 'test'}")

if __name__ == "__main__":
    main()


In [ ]:
{
  "model": {
    "type_map": ["Si","C","H"],
    "descriptor": {
      "type": "se_e2_a",
      "rcut": 6.0,
      "neuron": [25, 50, 100],
      "axis_neuron": 16,
      "resnet_dt": false
    },
    "fitting_net": {
      "neuron": [240, 240, 240],
      "resnet_dt": true
    }
  },
  "learning_rate": {
    "type": "exp",
    "start_lr": 1e-3,
    "decay_steps": 50000,
    "stop_lr": 1e-8
  },
  "loss": {
    "start_pref_e": 0.02, "limit_pref_e": 1.0,
    "start_pref_f": 1000.0, "limit_pref_f": 1.0,
    "start_pref_v": 0.0, "limit_pref_v": 0.1
  },
  "training": {
    "numb_steps": 300000,
    "save_freq": 5000,
    "disp_freq": 100,
    "training_data": {
      "systems": ["out_v1/deepmd_npy/train"],
      "batch_size": 1
    },
    "validation_data": {
      "systems": ["out_v1/deepmd_npy/val"],
      "batch_size": 1
    }
  }
}


In [ ]:
dp train input.json
dp train --restart model.ckpt input.json
dp train --init-model model.ckpt input.json
dp train --init-frz-model frozen_model.pb input.json
dp freeze -o frozen_model.pb
dp test --model frozen_model.pb --system out_v1/deepmd_npy/test


In [ ]:
import dpdata

# 1. Đọc dữ liệu (ví dụ từ LAMMPS dump hoặc DeepMD raw/npy)
d = dpdata.System("out_v1/deepmd_npy/test/Si56C56H52", fmt="deepmd/raw", type_map=["Si", "C", "H"])

# 2. Dự đoán bằng mô hình DeepMD
d_pred = d.predict(dp="frozen_model.pb")

# 3. In kết quả
print("Predicted energies shape:", d_pred["energies"].shape)
print("First 5 energies:", d_pred["energies"][:5])

# 4. Nếu muốn, lưu lại
d_pred.to("deepmd/npy", "predicted_Si56C56H52")

import dpdata

for comp in ["Si56C56H52", "Si56C56H55", "Si56C56H58"]:
    d = dpdata.System(f"out_v1/deepmd_npy/test/{comp}", fmt="deepmd/raw", type_map=["Si","C","H"])
    d_pred = d.predict(dp="frozen_model.pb")
    d_pred.to("deepmd/npy", f"pred_{comp}")
    print(f"[OK] Predicted {comp} -> saved to pred_{comp}")



In [ ]:
import dpdata
import numpy as np
from deepmd.infer import DeepPot

# Load model
dp = DeepPot("frozen_model.pb")

# Load test system
test_dir = "out_v1/deepmd_npy/test/Si56C56H52"
system = dpdata.LabeledSystem(test_dir, fmt="deepmd/raw")  # hoặc "deepmd/npy" tùy dữ liệu

# Predict
coords = system["coords"]
cells = system["cells"]
types = system["atom_types"]

E_pred = dp.eval(coords, cells, types)
E_true = system["energies"]
F_pred = dp.eval_force(coords, cells, types)
F_true = system["forces"]

# Sai số
mae_E = np.mean(np.abs(E_pred - E_true))
mae_F = np.mean(np.abs(F_pred - F_true))
print("Energy MAE:", mae_E)
print("Force MAE:", mae_F)


In [ ]:
# predict_single.py
import dpdata
import numpy as np
from deepmd.infer import DeepPot

MODEL = "frozen_model.pb"                 # model đã freeze
SRC   = "path/to/your/structure"          # ví dụ: POSCAR, traj.xyz, thư mục deepmd/raw hoặc deepmd/npy
FMT   = "vasp/poscar"                     # đổi tương ứng: "vasp/poscar", "vasp/contcar", "xyz", "siesta/aimd_output", "deepmd/raw", "deepmd/npy", ...

# 1) Load cấu trúc (không cần nhãn ⇒ System là đủ)
try:
    sys = dpdata.System(SRC, fmt=FMT)
except Exception:
    # nếu file có cả nhãn (AIMD) bạn vẫn có thể load thành LabeledSystem
    sys = dpdata.LabeledSystem(SRC, fmt=FMT)

# 2) Chuẩn bị input cho DeepPot
coords = sys["coords"]        # shape: [n_frames, n_atoms, 3]
cells  = sys["cells"]         # shape: [n_frames, 3, 3]
types  = sys["atom_types"]    # shape: [n_atoms]
# Lưu ý: Thứ tự nguyên tố phải khớp type_map khi train (vd ["Si","C","H"])

# 3) Nạp model & dự đoán
dp = DeepPot(MODEL)
E_pred = dp.eval(coords, cells, types)             # shape: [n_frames]
F_pred = dp.eval_force(coords, cells, types)       # shape: [n_frames, n_atoms, 3]
V_pred = dp.eval_virial(coords, cells, types)      # shape: [n_frames, 3, 3] (nếu cần)

# 4) Lưu kết quả
np.savetxt("pred_energy.csv", E_pred, delimiter=",")
np.save("pred_forces.npy", F_pred)      # lưu .npy để giữ nguyên shape
np.save("pred_virial.npy", V_pred)
print("Done. Wrote: pred_energy.csv, pred_forces.npy, pred_virial.npy")


In [ ]:
# predict_folder.py
import dpdata, numpy as np
from deepmd.infer import DeepPot
from pathlib import Path

MODEL = "frozen_model.pb"
ROOT  = Path(r"path/to/folder")  # thư mục chứa nhiều cấu trúc
PATTERN = "POSCAR"               # hoặc "*.xyz", "STRUCT_OUT", "output.out", ...

FMT = "vasp/poscar"              # đổi phù hợp pattern
OUT = Path("pred_out")
OUT.mkdir(exist_ok=True)

dp = DeepPot(MODEL)

for f in ROOT.rglob(PATTERN):
    try:
        try:
            sys = dpdata.System(str(f), fmt=FMT)
        except Exception:
            sys = dpdata.LabeledSystem(str(f), fmt=FMT)

        coords, cells, types = sys["coords"], sys["cells"], sys["atom_types"]
        E = dp.eval(coords, cells, types)
        F = dp.eval_force(coords, cells, types)

        stem = f.stem if f.is_file() else f.name
        np.savetxt(OUT/f"{stem}_E.csv", E, delimiter=",")
        np.save(OUT/f"{stem}_F.npy", F)
        print(f"[OK] {f} -> frames={coords.shape[0]}")
    except Exception as e:
        print(f"[WARN] Skip {f}: {e}")

print("Done. Results in pred_out/")
